# 05 - Monte Carlo Simulation & Efficient Frontier

**Financial Portfolio Tracker** | Notebook 5 of 5

---

## Objective

Project future portfolio value using Monte Carlo simulation based on Geometric Brownian Motion, then optimize the asset allocation using mean-variance optimization to find the efficient frontier.

## Mathematical Foundation

### Geometric Brownian Motion (GBM)

Asset prices are modeled as a stochastic process following GBM:

$$dS = \mu S\,dt + \sigma S\,dW$$

where $\mu$ is the drift (expected return), $\sigma$ is the volatility, and $dW$ is a Wiener process increment ($dW \sim N(0, dt)$).

### Ito's Lemma Derivation

Applying Ito's lemma to $\ln S$ yields the closed-form solution:

$$S_t = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t\right]$$

The term $-\frac{\sigma^2}{2}$ is the Ito correction -- it accounts for the difference between the expected arithmetic return $\mu$ and the geometric (compound) return. Without this correction, the simulation would systematically overestimate future values.

For discrete time steps:

$$S_{t+\Delta t} = S_t \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)\Delta t + \sigma \sqrt{\Delta t}\,\epsilon\right]$$

where $\epsilon \sim N(0, 1)$ is a standard normal random variable.

### Efficient Frontier

The efficient frontier traces the set of portfolios that maximize expected return for each level of risk (or equivalently, minimize risk for each target return). From Markowitz (1952), the portfolio variance is:

$$\sigma_p^2 = \mathbf{w}^\top \Sigma \mathbf{w}$$

where $\Sigma$ is the covariance matrix and $\mathbf{w}$ is the weight vector.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy import optimize, stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045
INITIAL_VALUE = 100_000

WEIGHTS = {
    "VOO": 0.30, "VXUS": 0.20, "VWO": 0.10,
    "BND": 0.20, "VNQ": 0.10, "GLD": 0.10,
}
TICKERS = list(WEIGHTS.keys())
BENCHMARK = "SPY"

CATEGORY_MAP = {
    "VOO": "US Equity", "VXUS": "International Equity", "VWO": "Emerging Markets",
    "BND": "Fixed Income", "VNQ": "Real Estate", "GLD": "Commodities",
}

In [ ]:
# Load data
returns = pd.read_parquet(PROCESSED_DIR / "returns.parquet")
portfolio_returns = returns["Portfolio"].dropna()
asset_returns = returns[TICKERS].dropna()

print(f"Loaded {len(portfolio_returns)} daily return observations")
print(f"Period: {portfolio_returns.index.min().date()} to {portfolio_returns.index.max().date()}")

## Part 1: Monte Carlo Simulation

### Parameter Estimation

We estimate the GBM parameters $\mu$ and $\sigma$ from historical daily returns.

In [ ]:
# GBM parameters from historical data
mu_daily = portfolio_returns.mean()
sigma_daily = portfolio_returns.std()

# Annualised parameters
mu_annual = mu_daily * TRADING_DAYS
sigma_annual = sigma_daily * np.sqrt(TRADING_DAYS)

# GBM drift (with Ito correction)
drift_daily = mu_daily - 0.5 * sigma_daily ** 2

print("GBM PARAMETER ESTIMATES")
print("=" * 50)
print(f"  Daily mu:     {mu_daily:.6f}")
print(f"  Daily sigma:  {sigma_daily:.6f}")
print(f"  Annual mu:    {mu_annual:.4f} ({mu_annual:.2%})")
print(f"  Annual sigma: {sigma_annual:.4f} ({sigma_annual:.2%})")
print(f"  Daily drift:  {drift_daily:.6f} (with Ito correction: -sigma^2/2 = {-0.5 * sigma_daily**2:.6f})")

### Simulation Engine

We generate 1,000 paths over 252 trading days (1 year forward), applying the GBM formula at each daily time step.

In [ ]:
# Monte Carlo simulation parameters
N_SIMULATIONS = 1_000
N_DAYS = 252  # 1 year forward

rng = np.random.default_rng(42)

# Generate random shocks
random_shocks = rng.normal(0, 1, (N_SIMULATIONS, N_DAYS))

# GBM log returns for each step
log_returns = drift_daily + sigma_daily * random_shocks

# Cumulative log returns (prepend 0 for starting point)
log_paths = np.cumsum(log_returns, axis=1)
log_paths = np.hstack([np.zeros((N_SIMULATIONS, 1)), log_paths])

# Convert to price paths
price_paths = INITIAL_VALUE * np.exp(log_paths)

# Final values
final_values = price_paths[:, -1]

print(f"Monte Carlo Simulation: {N_SIMULATIONS} paths x {N_DAYS} days")
print(f"Starting value: ${INITIAL_VALUE:,}")
print(f"\nFinal Value Distribution:")
print(f"  Mean:   ${final_values.mean():,.0f}")
print(f"  Median: ${np.median(final_values):,.0f}")
print(f"  5th %%:  ${np.percentile(final_values, 5):,.0f}")
print(f"  95th %%: ${np.percentile(final_values, 95):,.0f}")
print(f"  Min:    ${final_values.min():,.0f}")
print(f"  Max:    ${final_values.max():,.0f}")

### Fan Chart (Confidence Bands)

The fan chart shows percentile bands of simulated paths, giving a visual sense of the range of possible outcomes.

In [ ]:
# Compute percentile paths
percentiles = {}
for p in [5, 10, 25, 50, 75, 90, 95]:
    percentiles[p] = np.percentile(price_paths, p, axis=0)

days_axis = np.arange(N_DAYS + 1)

fig = go.Figure()

# 5-95 percentile band
fig.add_trace(go.Scatter(
    x=days_axis, y=percentiles[95],
    mode="lines", line=dict(width=0), showlegend=False,
))
fig.add_trace(go.Scatter(
    x=days_axis, y=percentiles[5],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor="rgba(31, 119, 180, 0.1)",
    name="5th-95th percentile",
))

# 25-75 percentile band
fig.add_trace(go.Scatter(
    x=days_axis, y=percentiles[75],
    mode="lines", line=dict(width=0), showlegend=False,
))
fig.add_trace(go.Scatter(
    x=days_axis, y=percentiles[25],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor="rgba(31, 119, 180, 0.25)",
    name="25th-75th percentile",
))

# Median path
fig.add_trace(go.Scatter(
    x=days_axis, y=percentiles[50],
    mode="lines",
    name="Median (50th)",
    line=dict(color="#1f77b4", width=2.5),
))

# Starting value reference
fig.add_hline(
    y=INITIAL_VALUE, line_dash="dash", line_color="gray", opacity=0.5,
    annotation_text=f"Initial: ${INITIAL_VALUE:,}",
)

# Sample paths (5 random paths for visual interest)
for i in rng.choice(N_SIMULATIONS, 5, replace=False):
    fig.add_trace(go.Scatter(
        x=days_axis, y=price_paths[i],
        mode="lines",
        line=dict(width=0.5, color="rgba(150, 150, 150, 0.3)"),
        showlegend=False,
    ))

fig.update_layout(
    title=f"Monte Carlo Simulation: {N_SIMULATIONS} Paths, {N_DAYS} Trading Days Forward",
    xaxis_title="Trading Days",
    yaxis_title="Portfolio Value (USD)",
    height=550,
    template="plotly_white",
    yaxis_tickformat="$,.0f",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

### Probability Calculations

Monte Carlo simulation lets us estimate the probability of achieving specific investment outcomes.

In [ ]:
# Probability calculations
prob_profit = np.mean(final_values > INITIAL_VALUE)
prob_10_gain = np.mean(final_values > INITIAL_VALUE * 1.10)
prob_20_gain = np.mean(final_values > INITIAL_VALUE * 1.20)
prob_10_loss = np.mean(final_values < INITIAL_VALUE * 0.90)
prob_20_loss = np.mean(final_values < INITIAL_VALUE * 0.80)

print("PROBABILITY ESTIMATES (1-Year Horizon)")
print("=" * 50)
print(f"  P(profit > 0%):    {prob_profit:.1%}")
print(f"  P(gain > 10%):     {prob_10_gain:.1%}")
print(f"  P(gain > 20%):     {prob_20_gain:.1%}")
print(f"  P(loss > 10%):     {prob_10_loss:.1%}")
print(f"  P(loss > 20%):     {prob_20_loss:.1%}")
print(f"")
print(f"  Expected value:    ${final_values.mean():,.0f}")
print(f"  Expected return:   {(final_values.mean() / INITIAL_VALUE - 1):+.2%}")
print(f"  Value at Risk (5%):${np.percentile(final_values, 5):,.0f} (worst case with 95% confidence)")

In [ ]:
# Distribution of final values
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=final_values,
    nbinsx=80,
    name="Final Portfolio Value",
    marker_color="rgba(31, 119, 180, 0.6)",
))

# Key reference lines
fig.add_vline(x=INITIAL_VALUE, line_dash="dash", line_color="gray",
              annotation_text=f"Initial: ${INITIAL_VALUE:,}")
fig.add_vline(x=np.median(final_values), line_dash="solid", line_color="#1f77b4",
              annotation_text=f"Median: ${np.median(final_values):,.0f}")
fig.add_vline(x=np.percentile(final_values, 5), line_dash="dash", line_color="#d62728",
              annotation_text=f"5th pct: ${np.percentile(final_values, 5):,.0f}")

fig.update_layout(
    title="Distribution of Simulated Final Portfolio Values (1-Year)",
    xaxis_title="Portfolio Value (USD)",
    yaxis_title="Frequency",
    height=450,
    template="plotly_white",
    xaxis_tickformat="$,.0f",
)

fig.show()

## Part 2: Efficient Frontier

### Random Portfolio Generation

We generate 5,000 random portfolio allocations to visualize the risk-return space, then use optimization to trace the true efficient frontier.

In [ ]:
# Annualised mean returns and covariance matrix
mean_returns = asset_returns.mean() * TRADING_DAYS
cov_matrix = asset_returns.cov() * TRADING_DAYS

print("ANNUALISED MEAN RETURNS")
print("=" * 40)
for t in TICKERS:
    print(f"  {t}: {mean_returns[t]:+.2%}")

print(f"\nANNUALISED COVARIANCE MATRIX")
print("=" * 40)
(cov_matrix * 100).round(4)

In [ ]:
# Generate 5,000 random portfolios
N_PORTFOLIOS = 5_000
n_assets = len(TICKERS)

rng = np.random.default_rng(42)

results_ret = np.zeros(N_PORTFOLIOS)
results_vol = np.zeros(N_PORTFOLIOS)
results_sharpe = np.zeros(N_PORTFOLIOS)
all_weights = np.zeros((N_PORTFOLIOS, n_assets))

for i in range(N_PORTFOLIOS):
    # Random weights that sum to 1
    w = rng.random(n_assets)
    w = w / w.sum()
    all_weights[i] = w
    
    # Portfolio return and risk
    port_ret = np.dot(w, mean_returns)
    port_vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix.values, w)))
    
    results_ret[i] = port_ret
    results_vol[i] = port_vol
    results_sharpe[i] = (port_ret - RISK_FREE_RATE) / port_vol if port_vol > 0 else 0

print(f"Generated {N_PORTFOLIOS:,} random portfolios")
print(f"Return range: {results_ret.min():.2%} to {results_ret.max():.2%}")
print(f"Volatility range: {results_vol.min():.2%} to {results_vol.max():.2%}")
print(f"Sharpe range: {results_sharpe.min():.3f} to {results_sharpe.max():.3f}")

### Optimal Portfolios

We find two special portfolios using `scipy.optimize`:
1. **Minimum Variance Portfolio**: Lowest possible risk
2. **Maximum Sharpe Portfolio**: Best risk-adjusted return (tangency portfolio)

In [ ]:
def portfolio_stats(weights):
    """Return (return, volatility) for a given weight vector."""
    ret = np.dot(weights, mean_returns)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix.values, weights)))
    return ret, vol

# --- Minimum Variance Portfolio ---
def min_vol_objective(w):
    return np.sqrt(np.dot(w.T, np.dot(cov_matrix.values, w)))

constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
bounds = tuple((0, 1) for _ in range(n_assets))
x0 = np.ones(n_assets) / n_assets

min_var_result = optimize.minimize(
    min_vol_objective, x0, method="SLSQP",
    bounds=bounds, constraints=constraints
)

min_var_weights = min_var_result.x
min_var_ret, min_var_vol = portfolio_stats(min_var_weights)
min_var_sharpe = (min_var_ret - RISK_FREE_RATE) / min_var_vol

print("MINIMUM VARIANCE PORTFOLIO")
print("=" * 50)
for i, t in enumerate(TICKERS):
    print(f"  {t}: {min_var_weights[i]:.1%}")
print(f"  Return:    {min_var_ret:.2%}")
print(f"  Volatility:{min_var_vol:.2%}")
print(f"  Sharpe:    {min_var_sharpe:.3f}")

In [ ]:
# --- Maximum Sharpe Portfolio ---
def neg_sharpe(w):
    ret = np.dot(w, mean_returns)
    vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix.values, w)))
    if vol == 0:
        return 0
    return -(ret - RISK_FREE_RATE) / vol

max_sharpe_result = optimize.minimize(
    neg_sharpe, x0, method="SLSQP",
    bounds=bounds, constraints=constraints
)

max_sharpe_weights = max_sharpe_result.x
max_sharpe_ret, max_sharpe_vol = portfolio_stats(max_sharpe_weights)
max_sharpe_val = (max_sharpe_ret - RISK_FREE_RATE) / max_sharpe_vol

print("MAXIMUM SHARPE PORTFOLIO")
print("=" * 50)
for i, t in enumerate(TICKERS):
    print(f"  {t}: {max_sharpe_weights[i]:.1%}")
print(f"  Return:    {max_sharpe_ret:.2%}")
print(f"  Volatility:{max_sharpe_vol:.2%}")
print(f"  Sharpe:    {max_sharpe_val:.3f}")

### Efficient Frontier Curve

We trace the efficient frontier by solving for the minimum-volatility portfolio at each target return level.

In [ ]:
# Trace the efficient frontier
n_frontier_points = 50
target_returns = np.linspace(min_var_ret, mean_returns.max(), n_frontier_points)

frontier_ret = []
frontier_vol = []

for target in target_returns:
    constraints_frontier = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w, t=target: np.dot(w, mean_returns) - t},
    ]
    
    result = optimize.minimize(
        min_vol_objective, x0, method="SLSQP",
        bounds=bounds, constraints=constraints_frontier,
        options={"maxiter": 1000, "ftol": 1e-10},
    )
    
    if result.success:
        vol = min_vol_objective(result.x)
        frontier_ret.append(target)
        frontier_vol.append(vol)

print(f"Traced {len(frontier_ret)} points on the efficient frontier")

In [ ]:
# --- Efficient Frontier Plot ---
fig = go.Figure()

# Random portfolios (scatter, colored by Sharpe)
fig.add_trace(go.Scatter(
    x=results_vol * 100, y=results_ret * 100,
    mode="markers",
    marker=dict(
        size=3,
        color=results_sharpe,
        colorscale="Viridis",
        colorbar=dict(title="Sharpe Ratio", x=1.02),
        showscale=True,
        opacity=0.5,
    ),
    name="Random Portfolios",
    text=[f"Sharpe: {s:.3f}" for s in results_sharpe],
))

# Efficient frontier curve
fig.add_trace(go.Scatter(
    x=[v * 100 for v in frontier_vol],
    y=[r * 100 for r in frontier_ret],
    mode="lines",
    name="Efficient Frontier",
    line=dict(color="#d62728", width=3),
))

# Current portfolio
current_w = np.array([WEIGHTS[t] for t in TICKERS])
current_ret, current_vol = portfolio_stats(current_w)
fig.add_trace(go.Scatter(
    x=[current_vol * 100], y=[current_ret * 100],
    mode="markers",
    marker=dict(size=15, color="#ff7f0e", symbol="star", line=dict(width=2, color="black")),
    name=f"Current Portfolio (Sharpe: {(current_ret - RISK_FREE_RATE) / current_vol:.3f})",
))

# Min variance portfolio
fig.add_trace(go.Scatter(
    x=[min_var_vol * 100], y=[min_var_ret * 100],
    mode="markers",
    marker=dict(size=15, color="#2ca02c", symbol="diamond", line=dict(width=2, color="black")),
    name=f"Min Variance (Sharpe: {min_var_sharpe:.3f})",
))

# Max Sharpe portfolio
fig.add_trace(go.Scatter(
    x=[max_sharpe_vol * 100], y=[max_sharpe_ret * 100],
    mode="markers",
    marker=dict(size=15, color="#9467bd", symbol="hexagon", line=dict(width=2, color="black")),
    name=f"Max Sharpe (Sharpe: {max_sharpe_val:.3f})",
))

# Individual assets
for t in TICKERS:
    fig.add_trace(go.Scatter(
        x=[asset_returns[t].std() * np.sqrt(TRADING_DAYS) * 100],
        y=[mean_returns[t] * 100],
        mode="markers+text",
        marker=dict(size=8, color="gray"),
        text=[t], textposition="top center",
        showlegend=False,
    ))

fig.update_layout(
    title="Efficient Frontier with Optimal Portfolios",
    xaxis_title="Annualised Volatility (%)",
    yaxis_title="Annualised Return (%)",
    height=600,
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, font=dict(size=10)),
)

fig.show()

### Weight Comparison: Current vs Optimal

How does the current allocation compare to the mathematically optimal portfolios?

In [ ]:
# Weight comparison table
weight_comparison = pd.DataFrame({
    "Ticker": TICKERS,
    "Category": [CATEGORY_MAP[t] for t in TICKERS],
    "Current": [WEIGHTS[t] for t in TICKERS],
    "Min Variance": min_var_weights,
    "Max Sharpe": max_sharpe_weights,
})

# Format as percentages
for col in ["Current", "Min Variance", "Max Sharpe"]:
    weight_comparison[col] = weight_comparison[col].round(4)

print("WEIGHT COMPARISON")
print("=" * 70)
weight_comparison

In [ ]:
# Grouped bar chart comparing weights
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Current Allocation",
    x=TICKERS, y=[WEIGHTS[t] * 100 for t in TICKERS],
    marker_color="#ff7f0e",
))
fig.add_trace(go.Bar(
    name="Min Variance",
    x=TICKERS, y=min_var_weights * 100,
    marker_color="#2ca02c",
))
fig.add_trace(go.Bar(
    name="Max Sharpe",
    x=TICKERS, y=max_sharpe_weights * 100,
    marker_color="#9467bd",
))

fig.update_layout(
    title="Portfolio Weight Comparison: Current vs Optimal Allocations",
    xaxis_title="Asset",
    yaxis_title="Weight (%)",
    barmode="group",
    height=450,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

### Portfolio Comparison Summary

In [ ]:
# Summary comparison of all three portfolios
current_sharpe = (current_ret - RISK_FREE_RATE) / current_vol

comparison = pd.DataFrame([
    {
        "Portfolio": "Current Allocation",
        "Ann. Return": f"{current_ret:.2%}",
        "Ann. Volatility": f"{current_vol:.2%}",
        "Sharpe Ratio": f"{current_sharpe:.3f}",
        "Top Holdings": ", ".join(f"{t} ({WEIGHTS[t]:.0%})" for t in sorted(WEIGHTS, key=WEIGHTS.get, reverse=True)[:3]),
    },
    {
        "Portfolio": "Min Variance (Optimal)",
        "Ann. Return": f"{min_var_ret:.2%}",
        "Ann. Volatility": f"{min_var_vol:.2%}",
        "Sharpe Ratio": f"{min_var_sharpe:.3f}",
        "Top Holdings": ", ".join(f"{TICKERS[i]} ({min_var_weights[i]:.0%})" for i in np.argsort(-min_var_weights)[:3]),
    },
    {
        "Portfolio": "Max Sharpe (Optimal)",
        "Ann. Return": f"{max_sharpe_ret:.2%}",
        "Ann. Volatility": f"{max_sharpe_vol:.2%}",
        "Sharpe Ratio": f"{max_sharpe_val:.3f}",
        "Top Holdings": ", ".join(f"{TICKERS[i]} ({max_sharpe_weights[i]:.0%})" for i in np.argsort(-max_sharpe_weights)[:3]),
    },
])

print("=" * 80)
print("PORTFOLIO COMPARISON")
print("=" * 80)
comparison

## Conclusions & Investment Recommendations

### Monte Carlo Findings

The 1,000-path GBM simulation reveals the distribution of possible outcomes for a $100,000 portfolio over one year. The median outcome is positive, but the fan chart shows meaningful downside risk in the lower percentiles. Investors should be prepared for potential short-term losses even with a diversified allocation.

### Efficient Frontier Insights

1. **Current allocation vs. optimal**: The current portfolio is positioned near (but not exactly on) the efficient frontier. The gap represents a potential improvement in risk-adjusted returns without changing the overall risk level.

2. **Minimum variance portfolio**: The optimizer tends to concentrate in BND (fixed income) and GLD (gold) to minimize total portfolio variance. This is mathematically optimal for risk minimization but may sacrifice too much return for most investors.

3. **Maximum Sharpe portfolio**: This allocation maximizes risk-adjusted returns. It typically overweights the highest-Sharpe individual assets while maintaining enough diversification to reduce volatility.

### Practical Recommendations

1. **Maintain diversification**: Even if the optimizer suggests concentrating in fewer assets, the out-of-sample stability of a diversified portfolio is valuable. Past correlations may not persist.

2. **Rebalance regularly**: Monthly rebalancing keeps the portfolio close to target weights and captures mean-reversion benefits.

3. **Monitor volatility regimes**: During high-volatility periods, consider temporarily shifting toward the minimum variance allocation. Return to the strategic allocation when volatility normalizes.

4. **Caveat**: Mean-variance optimization is sensitive to input estimates ("garbage in, garbage out"). Small changes in expected returns can lead to dramatically different optimal weights. The current allocation is a reasonable starting point that balances theoretical optimality with practical robustness.

---

*This analysis was completed as part of the Financial Portfolio Tracker project. The interactive dashboard at the Streamlit app applies these same calculations in real-time with live market data.*